In [54]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm         
import sympy as sp

In [55]:
calls = pd.read_parquet('data/processed/calls_clean_trainval.parquet')

In [56]:
calls['intrinsic'] = np.maximum(calls['underlying_last'] - calls['strike'], 0)

In [57]:
c0, r, T, S0, K, P0, delta, sigma = sp.symbols('C_0 r T S_0 K P_0 delta sigma')

In [58]:

def BSM_call(S0, K, T, r, sigma, delta):
    d1 = (np.log(S0/K) + (r - delta + 1/2 * sigma**2)*T)/(sigma*T)

    d2 = d1 - sigma*np.sqrt(T)

    c0 = np.exp(-r*T)*(S0*np.exp((r-delta)*T)*norm.cdf(d1) - K*norm.cdf(d2))
    return c0

In [59]:
calls.columns

Index(['quote_date', 'underlying_last', 'expire_date', 'dte', 'c_delta',
       'c_gamma', 'c_vega', 'c_theta', 'c_rho', 'c_iv', 'c_volume', 'c_last',
       'c_size', 'c_bid', 'c_ask', 'strike', 'mid', 'moneyness', 'year',
       'split', 'intrinsic'],
      dtype='str')

In [60]:
rates = pd.read_csv('https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS3MO')
rates = rates.rename(columns={'DGS3MO':'r'})
rates['r'] = rates['r']/100
rates.dropna().sort_values('observation_date', inplace=True)

rate_lookup = rates.set_index('observation_date')['r'].ffill() 
rate_lookup.index = pd.to_datetime(rate_lookup.index)
calls['r'] = calls['quote_date'].map(rate_lookup)


C:\Users\matth\AppData\Local\Temp\ipykernel_23680\3709423469.py:4: Pandas4Warning: The inplace keyword in DataFrame.sort_values is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  rates.dropna().sort_values('observation_date', inplace=True)


In [61]:
spx = calls.groupby('quote_date')['underlying_last'].first().sort_index()
logret = np.log(spx).diff()
sigma = logret.rolling(63).std() * np.sqrt(252)

calls['sigma'] = calls['quote_date'].map(sigma)

print(calls['sigma'].isna().sum(), 'rows without sigma')

calls = calls.dropna(subset=['sigma'])

27476 rows without sigma


In [62]:
calls['BSM'] = BSM_call(calls['underlying_last'], calls['strike'],calls['dte'],calls['r'], calls['sigma'], calls['c_delta'])

In [63]:
calls['BSM']

27476     -1.299103e-10
27477     -2.460474e-10
27478     -6.786789e-10
27479     -8.566358e-09
27480     -1.030733e-05
               ...     
9006165    7.112601e-37
9006166    5.660299e-30
9006167    7.191460e-24
9006168    1.480629e-19
9006169    7.732454e-08
Name: BSM, Length: 8978694, dtype: float64